### Legendre Polynomials

Legendre polynomials are the keys to understand Gauss-Legendre quadrature. In the field of numerical analysis, they are define as the set of polynomials $P_n(x)$ that satisfies:

$$
    \int_{-1}^{1} P_n(x) P_m(x) dx \triangleq 0 \quad \text{for } n \neq m \quad \text{Orthogonality in } [-1, 1]
$$

And:

$$
    P_n(x = 1) \triangleq 1 \quad \text{Normalization}
$$

Let's find them using symbolic Gram-Schmidt algorithm applied to the canonical polynomial basis $\left\{ e_i(x) \right\}$:

$$
    P_k(x) = e_k(x) - \sum_{j = 0}^{k} \, \frac{\langle P_j(x), e_k(x) \rangle}{\langle  P_j(x), P_j(x) \rangle} P_j(x) \quad \{ e_k(x) \} = \left\{ 1, x, x^2, x^3, x^4, ... \right\}
$$

With the normalization condition $P_k(1) = 1$

In [1]:
import sympy as smp
from sympy import legendre

In [2]:
def dot(pol_a, pol_b): # Computes the scalar product bewteen 2 polynomials in [-1, 1]
    global x
    return smp.integrate(pol_a * pol_b, (x, -1, 1))

def normalize(pol): # Makes a polynomial to be equal to 1 in x = 1
    global x
    return pol / pol.subs(x, 1)

In [3]:
x = smp.symbols('x', real = True) # Free variable

e0, e1, e2, e3 = [x ** i for i in range(4)] # Canonical basis of polynomials: 1, x, x^2, x^3, ...

In [4]:
p0 = e0 # P_0(x)
p0

1

In [5]:
p1 = e1 - p0 * (dot(p0, e1) / dot(p0, p0)) # P_1(x), already normalized
p1

x

In [6]:
p2 = e2 - p0 * (dot(p0, e2) / dot(p0, p0)) - p1 * (dot(e2, p1) / dot(p1, p1)) # P_2(x)
p2 = normalize(p2)
p2

3*x**2/2 - 1/2

In [7]:
p3 = e3 - p0 * (dot(p0, e3) / dot(p0, p0)) - p1 * (dot(p1, e3) / dot(p1, p1)) - p2 * (dot(p2, e3) / dot(p2, p2)) # P_3(x)
p3 = normalize(p3)
p3

5*x**3/2 - 3*x/2

In [8]:
legendre(0, x), legendre(1, x),legendre(2, x), legendre(3, x) # Exact results

(1, x, 3*x**2/2 - 1/2, 5*x**3/2 - 3*x/2)

Gauss-Legendre quadrature is:

$$
    \int_{-1}^{1} f(x) \, dx = \sum_{i = 1}^{N} w_i f(x_i) 
$$

Where:

- $N$ is the degree of the Legendre polynomial $P_N(x)$
- $w_i$ is the i-th weight
- $x_i$ is the i-th root of the Legendre polynomial $P_N(x)$

Weights $w_i$ can be computed using the following formula:

$$
    w_i = \frac{2}{\left( 1 - x_{i}^{2} \right) \left[ \dot{P_{N}}(x_i) \right]^{2}}
$$

Where $\dot{P_{N}}(x_i)$ is the derivative with respect to $x$ of the Legendre polynomial with degree $N$, evaluated at the i-th root $x_i$

Let's find a symbolic formulation for $N = 5$

In [9]:
p5 = legendre(5, x) # Legendre polynomial P_N(5)
p5

63*x**5/8 - 35*x**3/4 + 15*x/8

In [10]:
roots = smp.solve(p5, simplify = True) # Roots
roots

[0,
 -sqrt(245 - 14*sqrt(70))/21,
 sqrt(245 - 14*sqrt(70))/21,
 -sqrt(14*sqrt(70) + 245)/21,
 sqrt(14*sqrt(70) + 245)/21]

In [11]:
p5_diff = smp.diff(p5, x, 1) # Derivative of P_5(x)
p5_diff

315*x**4/8 - 105*x**2/4 + 15/8

In [12]:
# Computing weights using the formula
weights = []
for root in roots:
    w = 2 / ((1 - root ** 2) * (p5_diff.subs(x, root)) ** 2)
    w = w.simplify()
    weights.append(w)

weights

[128/225,
 13*sqrt(70)/900 + 161/450,
 13*sqrt(70)/900 + 161/450,
 161/450 - 13*sqrt(70)/900,
 161/450 - 13*sqrt(70)/900]

In [13]:
f = smp.Function('f', real = True)(x) # General function

I = 0
for weight, root in zip(weights, roots): # Gauss-Legendre formula N = 5
    I += weight * f.subs(x, root)

I.simplify() # Integral result for N = 5

128*f(0)/225 + (13*sqrt(70) + 322)*f(-sqrt(7)*sqrt(35 - 2*sqrt(70))/21)/900 + (13*sqrt(70) + 322)*f(sqrt(7)*sqrt(35 - 2*sqrt(70))/21)/900 + (322 - 13*sqrt(70))*f(-sqrt(7)*sqrt(2*sqrt(70) + 35)/21)/900 + (322 - 13*sqrt(70))*f(sqrt(7)*sqrt(2*sqrt(70) + 35)/21)/900

In general, Golub–Welsch algorithm computes weights and roots of $P_N(x)$ by solving an eigenvalue problem